In [ ]:
import networkx as nx
import random
import pandas as pd
import numpy as np
import random
import gzip
import csv
from pies import pies_sampling
from ties import ties_sampling
from scipy.stats import ks_2samp
random.seed(42)
np.random.seed(42)
# url https://www.researchgate.net/publication/254639513_Network_Sampling_via_Edge-based_Node_Selection_with_Graph_Induction

In [2]:
# demo data https://snap.stanford.edu/data/soc-sign-bitcoin-otc.html
filename = 'data/soc-sign-bitcoinotc.csv.gz'
data = []
with gzip.open(filename, 'rt') as f:
    for row in csv.reader(f):
        data.append(row)
data = np.array(data)
df = pd.DataFrame(data,columns=['source','target','rating','time'])
G = nx.DiGraph()
G.add_edges_from(df[['source','target']].values)
# add time and rating to the edges
for i in range(len(df)):
    G[df['source'][i]][df['target'][i]]['time'] = df['time'][i]
    G[df['source'][i]][df['target'][i]]['rating'] = int(df['rating'][i])

In [3]:
class Helper:


    def __init__(self):
        pass


    @staticmethod   
    def make_snapshots_nodes(G, N):
        """
        Creates snapshots of the graph based on the number of nodes.

        Args:
            G (nx.Graph or nx.DiGraph): The input graph.
            N (int): The number of snapshots to create.

        Returns:
            list of nx.Graph or nx.DiGraph: A list of graph snapshots.
        """
        # sort G on Time attribute
        G = nx.DiGraph(sorted(G.edges(data=True), key=lambda x: x[2]['time']))
        nodes = list(G.nodes)
        chunk_size = len(nodes) // N  # Base chunk size
        leftover = len(nodes) % N  # Number of extra nodes to distribute

        # print(f'Base chunk size {chunk_size}, Leftover {leftover}')
        
        # Split the graph into N snapshots
        snapshots = []
        snapshot_nodes = []
        for i in range(N):
            snapshot_nodes_local = nodes[i * chunk_size: (i + 1) * chunk_size]
            snapshot_nodes = snapshot_nodes_local + snapshot_nodes
            if i < leftover:
                snapshot_nodes.append(nodes[N * chunk_size + i])
            snapshot_graph = G.subgraph(snapshot_nodes)
            snapshots.append(snapshot_graph.copy())
        return snapshots
    
    @staticmethod
    def make_snapshots_edges(G, N):
        edges = list(G.edges(data=True))
        chunk_size = len(edges) // N
        print(f'chunk size {chunk_size}')
        edge_chunks = [edges[i * chunk_size: (i + 1) * chunk_size] for i in range(N)]
        leftover = len(edges) % N
        for i in range(leftover):
            edge_chunks[i].append(edges[N * chunk_size + i])
        snapshots = []
        snapshot_graph = nx.DiGraph()
        for chunk in edge_chunks:
            snapshot_graph.add_edges_from(chunk)
            snapshots.append(snapshot_graph.copy())
        return snapshots
    

    

In [ ]:
amount_of_snapshots = 4
Gt = Helper.make_snapshots_nodes(G,amount_of_snapshots)
fractions = [len(snapshot.nodes()) / len(G.nodes()) for snapshot in Gt]
St = [ties_sampling(G,fraction) for fraction in fractions]
#TODO check what to do with 100% sample at i = -1 because comparing 100% sample with 100% sample is not possible
fractions

[0.2501275293317463, 0.5002550586634926, 0.7500425097772487, 1.0]
[0.2501275293317463, 0.5000850195544976, 0.7500425097772487, 1.0]
[17290, 27124, 32624, 35592]
[8402, 18988, 28415, 35592]


[0.2501275293317463, 0.5000850195544976, 0.7500425097772487, 1.0]

In [5]:
# new benchmark ideas temproal
# Edge Lifetimes: Check the overlap in edges present across time.
# Node Activity: Compare the temporal activity patterns of nodes., see how nodes expand their list of neighbors
# Temporal Motifs: Analyze the frequency of temporal motifs (e.g., triadic closure over time). (ik dacht dat dit heel traag is)
# Entropy of Node Degrees: Compute entropy for degree distributions in both sets of graphs
# Wasserstein Distance?

# new benchmark ideas directed
# Edge Density Over Time: Compare the edge density of the graph over time.
# Pagerank
# HITS: identify hubs and authorities in the graph
# Flow Efficiency Measure the efficiency of information or resource transfer along directed paths:

In [6]:
# weights of the edges
list(G.edges(data=True))[:4]

[('6', '2', {'time': '1289241911.72836', 'rating': 4}),
 ('6', '5', {'time': '1289241941.53378', 'rating': 2}),
 ('6', '4', {'time': '1289770700.4293', 'rating': 2}),
 ('6', '7', {'time': '1290826367.17211', 'rating': 5})]

In [ ]:
class TemporalBenchmark:

    def __init__(self,Gt,St):
        self.Gt = Gt[:-1] # TODO overleggen, sampling 100% from nodes and comparing is a waste of time
        self.St = St[:-1]


    def b_flow_efficiency(self,G: nx.DiGraph) -> float:
        """
        Calculate the Flow Efficiency of a directed graph.
        
        Flow Efficiency is defined as the sum of the reciprocal shortest path lengths 
        between all pairs of nodes (i, j) in the graph, divided by the total possible 
        number of node pairs.
        
        Parameters:
            G (nx.DiGraph): A NetworkX directed graph.
        
        Returns:
            float: The flow efficiency of the graph.
        """
        # Number of nodes in the graph
        n = G.number_of_nodes()
        
        # Total possible pairs of nodes
        total_pairs = n * (n - 1)
        
        # Handle the case of an empty graph or a single node
        if total_pairs == 0:
            return 0.0
        
        # Calculate reciprocal of shortest path lengths
        efficiency_sum = 0
        for source in G.nodes():
            # Get shortest path lengths from the source node
            shortest_paths = nx.single_source_shortest_path_length(G, source)
            for target, length in shortest_paths.items():
                if source != target and length > 0:
                    efficiency_sum += 1 / length
        
        # Compute flow efficiency
        flow_efficiency = efficiency_sum / total_pairs
        return flow_efficiency
    
    def b_edge_density(self,G: nx.DiGraph) -> float:
        """
        Calculate the edge density of a directed graph.
        
        Edge density is defined as the ratio of the number of edges in the graph 
        to the total possible number of edges between all pairs of nodes.
        
        Parameters:
            G (nx.DiGraph): A NetworkX directed graph.
        
        Returns:
            float: The edge density of the graph.
        """
        # Number of nodes in the graph
        n = G.number_of_nodes()
        
        # Total possible pairs of nodes
        total_pairs = n * (n - 1)
        
        # Handle the case of an empty graph or a single node
        if total_pairs == 0:
            return 0.0
        
        # Calculate the edge density
        edge_density = G.number_of_edges() / total_pairs
        return edge_density
    

    def b_compute_weighted_hits_hubs(self,G: nx.DiGraph, weight='rating', max_iter=100, tol=1e-8):
        """
        Computes the HITS algorithm with weighted edges to calculate hub scores in a directed graph.

        Parameters:
            G (nx.DiGraph): A directed graph with weighted edges.
            weight (str): The edge attribute used as the weight.
            max_iter (int): Maximum number of iterations for convergence.
            tol (float): Convergence tolerance for changes in scores.

        Returns:
            dict: A dictionary where keys are nodes and values are their hub scores.
        """
        # Initialize hub and authority scores to 1
        hubs = {node: 1.0 for node in G}
        authorities = {node: 1.0 for node in G}
        
        for _ in range(max_iter):
            # Update authority scores based on weighted predecessors
            new_authorities = {
                node: sum(hubs[neighbor] * G[neighbor][node].get(weight, 1) for neighbor in G.predecessors(node))
                for node in G
            }
            
            # Update hub scores based on weighted successors
            new_hubs = {
                node: sum(new_authorities[neighbor] * G[node][neighbor].get(weight, 1) for neighbor in G.successors(node))
                for node in G
            }
            
            # Normalize the scores
            norm_authorities = sum(new_authorities.values())
            norm_hubs = sum(new_hubs.values())
            
            new_authorities = {node: score / norm_authorities for node, score in new_authorities.items()}
            new_hubs = {node: score / norm_hubs for node, score in new_hubs.items()}
            
            # Check for convergence
            if all(abs(new_hubs[node] - hubs[node]) < tol for node in G) and \
            all(abs(new_authorities[node] - authorities[node]) < tol for node in G):
                break
            
            hubs = new_hubs
            authorities = new_authorities
        # create dict with key the node and as value a dict with the hub score and authority score
        dict_all = {}
        for node in G:
            dict_all[node] = {'hub': hubs[node], 'authority': authorities[node]}
        return dict_all
    

    def T1(self):
        """
        Edge Lifetimes: Check the overlap in edges present across time.
        """
        efficiency_g = [self.b_flow_efficiency(g_t) for g_t in self.Gt]
        efficiency_s = [self.b_flow_efficiency(g_s) for g_s in self.St]
        statistic  = ks_2samp(efficiency_g, efficiency_s).statistic
        return statistic
    

    def T2(self):
        """
        Edge Density Over Time: Compare the edge density of the graph over time.
        """
        density_g = [self.b_edge_density(g_t) for g_t in self.Gt]
        density_s = [self.b_edge_density(g_s) for g_s in self.St]
        statistic = ks_2samp(density_g, density_s).statistic
        return statistic
    
    def T3(self):
        """
        Hub Scores: Compare the hub scores of the graphs.
        """
        hubs_authorities_g_dict = [benchmark.b_compute_weighted_hits_hubs(g_t) for g_t in Gt]
        hubs_authorities_s_dict = [benchmark.b_compute_weighted_hits_hubs(g_s) for g_s in St]
        hubs_g_dict_all = {}
        hubs_s_dict_all = {}
        for hubs in hubs_authorities_g_dict:
            for key, value in hubs.items():
                if key in hubs_g_dict_all:
                    hubs_g_dict_all[key].append(value)
                else:
                    hubs_g_dict_all[key] = [value]
        for hubs in hubs_authorities_s_dict:
            for key, value in hubs.items():
                if key in hubs_s_dict_all:
                    hubs_s_dict_all[key].append(value)
                else:
                    hubs_s_dict_all[key] = [value]

        # now instead of a list get the mean of the values
        hubs_g = {}
        for key,value in hubs_g_dict_all.items():
            hubs_g[key] = {'hub': np.mean([x['hub'] for x in value]), 'authority': np.mean([x['authority'] for x in value])}
        hubs_s = {}
        for key,value in hubs_s_dict_all.items():
            hubs_s[key] = {'hub': np.mean([x['hub'] for x in value]), 'authority': np.mean([x['authority'] for x in value])}
        statistic_hubs = ks_2samp([x['hub'] for x in hubs_g.values()], [x['hub'] for x in hubs_s.values()]).statistic
        statistic_authorities = ks_2samp([x['authority'] for x in hubs_g.values()], [x['authority'] for x in hubs_s.values()]).statistic
        return {'statistic_hubs':statistic_hubs, 'statistic_authorities':statistic_authorities}

    def T4(self):
        """
        Compare PageRank distributions between real and sampled nodes using KS statistic.
        # first get the pagerank for each node over time
        # then get the ks statistic and p-value for each node
        # at last get the mean of the ks statistic and p-value per node
        Returns:
            dict: KS statistic and p-value for each node.
        """
        # Compute PageRank for real and sampled graphs
        real_pagerank = [nx.pagerank(graph) for graph in self.Gt]
        sampled_pagerank = [nx.pagerank(graph) for graph in self.St]
  
        real_scores = {}    # over time so K times for each slice
        sampled_scores = {} # over time so K times for each slice
        
        for t in range(len(real_pagerank)):
            for node in real_pagerank[t]:
                if node in real_scores:
                    real_scores[node].append(real_pagerank[t][node])
                else:
                    real_scores[node] = [real_pagerank[t][node]]
            for node in sampled_pagerank[t]:
                if node in sampled_scores:
                    sampled_scores[node].append(sampled_pagerank[t][node])
                else:
                    sampled_scores[node] = [sampled_pagerank[t][node]]
        
        # get ks statistic and p-value for each node
        ks_statistic = {}
        for node in real_scores:
            if node in sampled_scores:
                ks_statistic[node]= ks_2samp(real_scores[node], sampled_scores[node]).statistic
        ks_statistic = np.mean(list(ks_statistic.values()))
        return ks_statistic

    def T5(self):
        """
        Betweenness Centrality: Compare the betweenness centrality of the graphs.
        """
        betweenness_g = [nx.betweenness_centrality(g_t,weight='rating',k=int(len(g_t.nodes())*0.05)) for g_t in self.Gt]
        betweenness_s = [nx.betweenness_centrality(g_s,weight='rating',k=int(len(g_s.nodes())*0.05)) for g_s in self.St]

        betweenness_g_dict_all = {}
        betweenness_s_dict_all = {}
        for betweenness in betweenness_g:
            for key, value in betweenness.items():
                if key in betweenness_g_dict_all:
                    betweenness_g_dict_all[key].append(value)
                else:
                    betweenness_g_dict_all[key] = [value]
        for betweenness in betweenness_s:
            for key, value in betweenness.items():
                if key in betweenness_s_dict_all:
                    betweenness_s_dict_all[key].append(value)
                else:
                    betweenness_s_dict_all[key] = [value]
        # now instead of a list get the mean of the values
        betweenness_g = {}
        for key,value in betweenness_g_dict_all.items():
            betweenness_g[key] = np.mean(value)
        betweenness_s = {}
        for key,value in betweenness_s_dict_all.items():
            betweenness_s[key] = np.mean(value)

        statistic = ks_2samp(list(betweenness_g.values()), list(betweenness_s.values())).statistic
        return statistic
benchmark = TemporalBenchmark(Gt,St)

In [8]:
statistic = benchmark.T1()
print(f"Flow Efficiency of the graph: {statistic:.4f}")

Flow Efficiency of the graph: 0.6667


In [9]:
statistic = benchmark.T2()
print(f"Edge Density of the graph: {statistic:.4f}")

Edge Density of the graph: 0.3333


In [10]:
benchmark_t3 = benchmark.T3()
print(benchmark_t3)

{'statistic_hubs': 0.16731848325114776, 'statistic_authorities': 0.2644108144873321}


In [11]:
benchmark_t4 = benchmark.T4()
print(benchmark_t4)

0.741051603161915


In [12]:
statistic = benchmark.T5()
print(f"Betweenness Centrality k-statisic: {statistic:.4f}")

Betweenness Centrality k-statisic: 0.1941
